In [ ]:
import pandas as pd
import numpy as np

session_df = pd.read_csv("../data/analysis_table/session_table.csv")
user_df = pd.read_csv("../data/analysis_table/user_table.csv")
funnel = pd.read_csv("../data/analysis_table/funnel_table.csv")
event_df = pd.read_csv("../data/analysis_table/data_cleaned.csv")

In [102]:
# define "Cart Abandonment": when a session has a cart event but didn't have purchase event.
abandoned = (session_df['cart_count'] > 0) & (session_df['purchase_count'] == 0)

total_sessions = len(session_df)

cart_sessions = (session_df['cart_count'] > 0).sum()
purchase_sessions = (session_df['purchase_count'] > 0).sum()
abandoned_sessions = abandoned.sum()

# calculate Cart-to-purchase rate and Abandonment rate
cart_to_purchase_rate = purchase_sessions / cart_sessions
abandonment_rate = abandoned_sessions / cart_sessions

print(cart_to_purchase_rate)

1.7160096540627514


In [103]:
# First of all, only taking the "view" events, since users see the prices in "view" events.
view_df = event_df[event_df['event_type'] == 'view'].copy()

session_price = (view_df.groupby('user_session')
                 .agg(session_avg_price = ('price', 'mean'))
                 .reset_index()
                 )

session_df = session_df.merge(session_price, on = 'user_session', how = 'left')

bins = [0, 50, 100, 500, 1000, 2000]
labels = ['0-50', '50-100', '100-500', '500-1000', '1000+']

session_df['price_bucket'] = pd.cut(session_df['session_avg_price'], bins = bins, labels = labels)

session_df[session_df['has_purchase'] > 0].head()

,Unnamed: 0,user_id,user_session,session_start,session_end,total_events,view_count,cart_count,purchase_count,total_revenue,has_view,has_cart,has_purchase,session_avg_price,price_bucket
44,44,457360398,b7360dbb-427a-428b-a4fb-1f8872884d4d,2019-10-01 04:05:12+00:00,2019-10-01 04:22:53+00:00,14,13,0,1,51.46,1,0,1,37.622308,0-50
170,170,504429960,54243b1c-14aa-4c29-9ddd-6607865700a1,2019-10-01 03:41:28+00:00,2019-10-01 04:14:48+00:00,45,44,0,1,275.66,1,0,1,350.288864,100-500
216,216,512365496,d5aa25d2-afb4-4644-8edc-d8004c03091d,2019-10-01 03:57:17+00:00,2019-10-01 03:58:32+00:00,4,3,0,1,1415.48,1,0,1,1415.480000,1000+
247,247,512371939,7737b8d9-9359-420e-8c91-2fa5b1c89279,2019-10-01 04:32:12+00:00,2019-10-01 04:34:23+00:00,2,1,0,1,36.55,1,0,1,36.550000,0-50
254,254,512373792,8b6814cb-7c0e-4cc9-8224-3a75672affff,2019-10-01 02:45:13+00:00,2019-10-01 02:53:14+00:00,12,11,0,1,213.61,1,0,1,213.610000,100-500


In [104]:
price_analysis = (session_df.groupby('price_bucket')
                  .agg(sessions = ('user_session', 'count'),
                       cart_rate = ('has_cart', 'mean'),
                       purchase_rate = ('has_purchase', 'mean')
                  ).reset_index()
                )

print(price_analysis)

  price_bucket  sessions  cart_rate  purchase_rate
0         0-50      5738   0.015685       0.054026
1       50-100      5103   0.016657       0.054282
2      100-500     16761   0.047551       0.070163
3     500-1000      4678   0.039119       0.052587
4        1000+      1921   0.044768       0.056221


/var/folders/j4/_7mlrdbn2_b0x18qdr4_qxr00000gn/T/ipykernel_5074/3119019423.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  price_analysis = (session_df.groupby('price_bucket')
